# NC Width Sweep + Threshold Prediction
**Google Colab A100**

**Setup:** Runtime → Change runtime type → **A100 GPU**

**What this notebook does:**

**Experiment 1 — Width sweep** (12 runs, ~90 min)

Tests whether fn at T_NC scales with network width.
If fn ∝ width^α for some exponent α, the threshold is not
arbitrary — it's set by model capacity.

| Width | Params | Expected fn |
|---|---|---|
| 128 | 0.15M | ~0.5? |
| 256 | 0.53M | ~0.7? |
| 512 | 2.10M | 1.063 (known) |
| 1024 | 8.39M | ~1.5? |

**Experiment 2 — Threshold prediction** (no new training)

For each run, measures the gap (epochs) between fn crossing
the predicted threshold and NC1 actually collapsing.
If this gap is consistent across widths, the threshold has
genuine predictive power — not just correlation.

**Outputs:** `sweep_width.csv`, `prediction_analysis.csv`,
`fig_width_sweep.png`, per-run CSVs

**Est. runtime: ~90 min** (A100, torch.compile, batch=512)

In [1]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from google.colab import files
from scipy import stats

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE = 'cuda'
assert torch.cuda.is_available(), 'No GPU — Runtime -> Change runtime type -> A100'
print(f'GPU:   {torch.cuda.get_device_name(0)}')
print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
print(f'Torch: {torch.__version__}')

# Previous confirmed thresholds for reference
BASELINE_W512 = {'fn': 1.063, 'T_NC': 310}  # depth=5, ReLU, wd=1e-4, seed=0
print(f'Baseline (width=512): fn={BASELINE_W512["fn"]}  T_NC={BASELINE_W512["T_NC"]}')
print()
print('Width sweep: [128, 256, 512, 1024]  x  3 seeds  =  12 runs')
print('Est. time: ~90 min on A100')
print('Width 512 already done — re-running for 3-seed average')


GPU:   NVIDIA A100-SXM4-40GB
VRAM:  42 GB
Torch: 2.10.0+cu128
Baseline (width=512): fn=1.063  T_NC=310

Width sweep: [128, 256, 512, 1024]  x  3 seeds  =  12 runs
Est. time: ~90 min on A100
Width 512 already done — re-running for 3-seed average


In [2]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,),(0.3081,))])
trainset  = torchvision.datasets.MNIST('/tmp/data', train=True,
                                       download=True, transform=transform)
testset   = torchvision.datasets.MNIST('/tmp/data', train=False,
                                       download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=512, shuffle=True,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=1024, shuffle=False,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
print(f'MNIST: {len(trainset):,} train / {len(testset):,} test')


100%|██████████| 9.91M/9.91M [00:00<00:00, 38.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.10MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.66MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.3MB/s]

MNIST: 60,000 train / 10,000 test


In [3]:
class MLP(nn.Module):
    """Depth-5 MLP with variable width. ReLU throughout."""
    def __init__(self, width=512, num_classes=10):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(784, width), nn.ReLU()]
        for _ in range(4):   # 4 more hidden = 5 total nonlinear layers
            layers += [nn.Linear(width, width), nn.ReLU()]
        self.body   = nn.Sequential(*layers)
        self.head   = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()
    def param_count(self): return sum(p.numel() for p in self.parameters())

# Show param counts for all widths
print('Width -> Parameters:')
for w in [128, 256, 512, 1024]:
    m = MLP(w)
    print(f'  width={w:>4}: {m.param_count()/1e6:.3f}M params')
del m


Width -> Parameters:
  width= 128: 0.168M params
  width= 256: 0.467M params
  width= 512: 1.458M params
  width=1024: 5.012M params


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)))
        ll.append(y.to(DEVICE, non_blocking=True))
    H = torch.cat(fl); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool, device=DEVICE)
    nc2  = (cos[mask]-(-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().to(DEVICE), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1':nc1, 'nc2':nc2, 'nc3':nc3,
            'feat_norm':H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x, y in loader:
            x,y = x.to(DEVICE,non_blocking=True), y.to(DEVICE,non_blocking=True)
            correct += (model(x).argmax(1)==y).sum().item()
            total   += len(y)
    return correct/total

print('Metrics ready (GPU-resident).')


Metrics ready (GPU-resident).


In [5]:
def run(model, name, lr=1e-3, wd=1e-4,
        phase1=200, phase2=500, nc_every=10, nc_thresh=0.01):
    try:
        model = torch.compile(model, mode='reduce-overhead')
    except Exception:
        pass
    model = model.to(DEVICE)
    K=10; rows=[]; terminal=False; t0=time.time()

    for phase, loss_fn, n_ep in [(1,'ce',phase1),(2,'mse',phase2)]:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
        off = phase1 if phase==2 else 0
        for ep_l in range(1, n_ep+1):
            ep = off + ep_l
            model.train()
            for x, y in train_loader:
                x,y = x.to(DEVICE,non_blocking=True), y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                loss = (F.mse_loss(logits, F.one_hot(y,K).float())
                        if loss_fn=='mse' else F.cross_entropy(logits, y))
                loss.backward(); opt.step()
            sch.step()

            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal ep={ep}')
                nc = (compute_nc(model, train_loader) if terminal
                      else {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None})
                rows.append({'epoch':ep,'phase':phase,'train':tr,'test':te,**nc})
                nc1s = f"{nc['nc1']:.5f}" if nc['nc1'] is not None else 'N/A'
                fns  = f"{nc['feat_norm']:.3f}" if nc['feat_norm'] else 'N/A'
                print(f'  ep={ep:>4} tr={tr:.4f} nc1={nc1s} '
                      f'fn={fns} t={(time.time()-t0)/60:.1f}m')
                if nc['nc1'] is not None and nc['nc1'] < nc_thresh:
                    print(f'  *** T_NC={ep}  fn={nc["feat_norm"]:.4f}')
                    return pd.DataFrame(rows), ep, nc['feat_norm']
    return pd.DataFrame(rows), None, None

print('run() ready.')


run() ready.


In [6]:
# Width sweep: 4 widths x 3 seeds = 12 runs
# depth=5, ReLU, wd=1e-4, lr=1e-3 — everything else fixed
# Width 512 seed 0 = baseline (T_NC=310, fn=1.063) — included for completeness

WIDTHS = [128, 256, 512, 1024]
width_results = []

for width in WIDTHS:
    for seed in range(3):
        name = f'w{width}-s{seed}'
        print(f'\n=== width={width}  seed={seed} ===')
        torch.manual_seed(seed)
        model = MLP(width=width)
        df, t_nc, fn = run(model, name, lr=1e-3, wd=1e-4)
        df.to_csv(f'/tmp/width{width}_s{seed}.csv', index=False)
        fn_val = float(fn) if fn is not None else None
        width_results.append({
            'width': width, 'seed': seed,
            'T_NC': t_nc, 'fn': fn_val,
            'test_acc': df.test.iloc[-1]
        })
        status = f'T_NC={t_nc}  fn={fn_val:.4f}' if t_nc else 'DNF'
        print(f'  => {status}')

df_width = pd.DataFrame(width_results)
df_width.to_csv('/tmp/sweep_width.csv', index=False)

print('\n=== WIDTH SWEEP SUMMARY ===')
g = df_width.dropna(subset=['fn']).groupby('width')['fn'].agg(['mean','std','count'])
g['cv'] = g['std'] / g['mean']
print(g.to_string())



=== width=128  seed=0 ===
  [w128-s0] Terminal ep=10
  ep=  10 tr=0.9939 nc1=0.49872 fn=40.778 t=0.8m
  ep=  20 tr=0.9961 nc1=0.44565 fn=40.426 t=1.5m
  ep=  30 tr=0.9982 nc1=0.38816 fn=38.065 t=2.1m
  ep=  40 tr=0.9972 nc1=0.35026 fn=34.798 t=2.7m
  ep=  50 tr=0.9983 nc1=0.30595 fn=30.189 t=3.4m
  ep=  60 tr=0.9996 nc1=0.26481 fn=29.725 t=4.0m
  ep=  70 tr=1.0000 nc1=0.22831 fn=28.803 t=4.6m
  ep=  80 tr=1.0000 nc1=0.22909 fn=28.999 t=5.3m
  ep=  90 tr=1.0000 nc1=0.19092 fn=26.136 t=5.9m
  ep= 100 tr=1.0000 nc1=0.20129 fn=26.468 t=6.6m
  ep= 110 tr=0.9998 nc1=0.18494 fn=22.005 t=7.2m
  ep= 120 tr=1.0000 nc1=0.18748 fn=24.855 t=7.9m
  ep= 130 tr=1.0000 nc1=0.19807 fn=24.990 t=8.5m
  ep= 140 tr=1.0000 nc1=0.17814 fn=22.447 t=9.1m
  ep= 150 tr=1.0000 nc1=0.18875 fn=24.116 t=9.8m
  ep= 160 tr=1.0000 nc1=0.19406 fn=24.169 t=10.4m
  ep= 170 tr=1.0000 nc1=0.19702 fn=24.225 t=11.1m
  ep= 180 tr=1.0000 nc1=0.19823 fn=24.250 t=11.7m
  ep= 190 tr=1.0000 nc1=0.19880 fn=24.255 t=12.4m
  ep= 200 t

W0327 22:02:57.500000 2225 torch/_dynamo/convert_frame.py:1676] [0/8] torch._dynamo hit config.recompile_limit (8)
W0327 22:02:57.500000 2225 torch/_dynamo/convert_frame.py:1676] [0/8]    function: 'forward' (/tmp/ipykernel_2225/1294726894.py:17)
W0327 22:02:57.500000 2225 torch/_dynamo/convert_frame.py:1676] [0/8]    last reason: 0/7: GLOBAL_STATE changed: grad_mode 
W0327 22:02:57.500000 2225 torch/_dynamo/convert_frame.py:1676] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0327 22:02:57.500000 2225 torch/_dynamo/convert_frame.py:1676] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/compile/programming_model.recompilation.html


  [w1024-s0] Terminal ep=10
  ep=  10 tr=0.9957 nc1=0.20382 fn=29.156 t=0.7m
  ep=  20 tr=0.9973 nc1=0.16373 fn=26.745 t=1.3m
  ep=  30 tr=0.9961 nc1=0.14873 fn=23.270 t=2.0m
  ep=  40 tr=0.9967 nc1=0.13684 fn=22.262 t=2.6m
  ep=  50 tr=0.9970 nc1=0.13651 fn=20.436 t=3.2m
  ep=  60 tr=0.9990 nc1=0.11854 fn=19.769 t=3.9m
  ep=  70 tr=0.9993 nc1=0.10480 fn=17.988 t=4.5m
  ep=  80 tr=0.9994 nc1=0.10001 fn=16.718 t=5.2m
  ep=  90 tr=1.0000 nc1=0.07806 fn=16.743 t=5.8m
  ep= 100 tr=0.9994 nc1=0.09448 fn=15.374 t=6.4m
  ep= 110 tr=1.0000 nc1=0.07963 fn=15.376 t=7.1m
  ep= 120 tr=1.0000 nc1=0.08635 fn=15.568 t=7.7m
  ep= 130 tr=1.0000 nc1=0.07782 fn=15.226 t=8.4m
  ep= 140 tr=1.0000 nc1=0.08470 fn=15.124 t=9.0m
  ep= 150 tr=1.0000 nc1=0.09240 fn=15.439 t=9.6m
  ep= 160 tr=1.0000 nc1=0.09849 fn=15.709 t=10.3m
  ep= 170 tr=1.0000 nc1=0.10421 fn=16.153 t=10.9m
  ep= 180 tr=1.0000 nc1=0.10877 fn=16.576 t=11.6m
  ep= 190 tr=1.0000 nc1=0.11185 fn=16.850 t=12.2m
  ep= 200 tr=1.0000 nc1=0.11234 fn=16

In [7]:
# ── Does fn scale with width? ────────────────────────────────────────────
# Test three scaling hypotheses:
#   (a) fn ∝ width^0.5  (square root)
#   (b) fn ∝ width^1    (linear)
#   (c) fn constant      (no scaling)

confirmed = df_width.dropna(subset=['fn'])
g = confirmed.groupby('width')['fn'].mean().reset_index()
widths = g.width.values.astype(float)
fns    = g.fn.values

print('=== SCALING ANALYSIS ===\n')

# Fit log-log: log(fn) = alpha * log(width) + const
log_w = np.log(widths)
log_f = np.log(fns)
slope, intercept, r, p, se = stats.linregress(log_w, log_f)
print(f'Log-log fit:  fn ~ width^{slope:.3f}')
print(f'  R²={r**2:.4f}  p={p:.4f}  se={se:.4f}')
print(f'  Implied scaling: fn ∝ width^{slope:.3f}')
print()

# Specific hypothesis tests
for exp, label in [(0.5,'sqrt'), (1.0,'linear'), (0.0,'constant')]:
    # Normalise by width^exp and check CV
    normalised = fns / (widths**exp)
    cv = normalised.std() / normalised.mean()
    print(f'  fn / width^{exp:.1f}: mean={normalised.mean():.4f}  '
        f'CV={cv:.4f}  ({'lowest CV = best fit' if cv < 0.1 else ''})')

print()
print('=== PREDICTION EXPERIMENT ===')
print('For each run, find epoch when fn first crosses threshold,')
print('then measure epochs until NC1 < 0.01')
print()

# Threshold prediction: does fn crossing predict T_NC?
# For each width, compute expected threshold from scaling law
pred_results = []
for width in WIDTHS:
    # Expected threshold from fitted scaling law
    thresh_pred = np.exp(intercept) * (width ** slope)
    print(f'width={width}: predicted fn threshold = {thresh_pred:.4f}')

    for seed in range(3):
        path = f'/tmp/width{width}_s{seed}.csv'
        try:
            df_s = pd.read_csv(path)
            nc_s = df_s.dropna(subset=['nc1'])
            if len(nc_s) == 0: continue

            # Find first epoch where fn <= thresh_pred
            cross = nc_s[nc_s.feat_norm <= thresh_pred]
            ep_cross = int(cross.epoch.iloc[0]) if len(cross) else None

            # Find T_NC
            t_rows = nc_s[nc_s.nc1 < 0.01]
            t_nc   = int(t_rows.epoch.iloc[0]) if len(t_rows) else None

            if ep_cross and t_nc:
                gap = t_nc - ep_cross
                print(f'  seed={seed}: fn crossed at ep={ep_cross}  '
                      f'T_NC={t_nc}  gap={gap} epochs')
                pred_results.append({'width':width,'seed':seed,
                                     'ep_cross':ep_cross,'T_NC':t_nc,
                                     'gap':gap})
            elif t_nc:
                print(f'  seed={seed}: fn never crossed threshold  T_NC={t_nc}')
            else:
                fn_final = nc_s.feat_norm.iloc[-1]
                print(f'  seed={seed}: DNF  fn_final={fn_final:.4f}')
        except FileNotFoundError:
            pass

if pred_results:
    df_pred = pd.DataFrame(pred_results)
    df_pred.to_csv('/tmp/prediction_analysis.csv', index=False)
    mean_gap = df_pred.gap.mean()
    std_gap  = df_pred.gap.std()
    cv_gap   = std_gap / mean_gap if mean_gap > 0 else float('inf')
    print(f'\nGap (ep_cross -> T_NC):')
    print(f'  mean={mean_gap:.1f}  std={std_gap:.1f}  CV={cv_gap:.3f}')
    if cv_gap < 0.3:
        print('  GAP IS CONSISTENT: crossing threshold predicts T_NC!')
    else:
        print(f'  Gap is variable (CV={cv_gap:.3f}) — threshold predicts'
              f' direction but not exact timing')


=== SCALING ANALYSIS ===

Log-log fit:  fn ~ width^-0.064
  R²=0.8383  p=0.0844  se=0.0199
  Implied scaling: fn ∝ width^-0.064

  fn / width^0.5: mean=0.0655  CV=0.4365  ()
  fn / width^1.0: mean=0.0043  CV=0.7701  ()
  fn / width^0.0: mean=1.1352  CV=0.0555  (lowest CV = best fit)

=== PREDICTION EXPERIMENT ===
For each run, find epoch when fn first crosses threshold,
then measure epochs until NC1 < 0.01

width=128: predicted fn threshold = 1.2115
  seed=0: fn crossed at ep=260  T_NC=400  gap=140 epochs
  seed=1: fn never crossed threshold  T_NC=370
  seed=2: fn never crossed threshold  T_NC=380
width=256: predicted fn threshold = 1.1589
  seed=0: fn crossed at ep=280  T_NC=320  gap=40 epochs
  seed=1: fn crossed at ep=260  T_NC=320  gap=60 epochs
  seed=2: fn crossed at ep=290  T_NC=340  gap=50 epochs
width=512: predicted fn threshold = 1.1087
  seed=0: fn crossed at ep=240  T_NC=320  gap=80 epochs
  seed=1: fn crossed at ep=240  T_NC=300  gap=60 epochs
  seed=2: fn crossed at ep=25

In [8]:
plt.rcParams.update({'font.family':'serif','font.size':11,
    'axes.spines.top':False,'axes.spines.right':False})

confirmed = df_width.dropna(subset=['fn'])
g_fn   = confirmed.groupby('width')['fn'].agg(['mean','std'])
g_tnc  = confirmed.dropna(subset=['T_NC']).groupby('width')['T_NC'].agg(['mean','std'])
widths = g_fn.index.values.astype(float)
fns    = g_fn['mean'].values

# Fit
log_w  = np.log(widths)
log_f  = np.log(fns)
slope, intercept, r, _, _ = stats.linregress(log_w, log_f)
w_fit  = np.linspace(100, 1200, 200)
fn_fit = np.exp(intercept) * w_fit**slope

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) fn at T_NC vs width — the scaling law
ax = axes[0]
ax.errorbar(widths, fns, yerr=g_fn['std'].values,
            fmt='o', color='#E91E63', ms=10, capsize=6,
            label='Data (mean ± std)', zorder=4)
ax.plot(w_fit, fn_fit, '--', color='#E91E63', lw=1.5,
        label=f'Fit: fn $\propto$ width$^{{{slope:.2f}}}$\n$R^2={r**2:.3f}$')
ax.set_xscale('log', base=2)
ax.set_xticks(widths.astype(int))
ax.set_xticklabels(widths.astype(int))
ax.set(xlabel='Width', ylabel='fn at $T_{NC}$',
       title='(a) Feature norm threshold scales with width')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# (b) T_NC vs width
ax = axes[1]
ax.errorbar(widths, g_tnc['mean'].values, yerr=g_tnc['std'].values,
            fmt='s-', color='#2196F3', ms=9, capsize=6, lw=2)
ax.set_xscale('log', base=2)
ax.set_xticks(widths.astype(int))
ax.set_xticklabels(widths.astype(int))
ax.set(xlabel='Width', ylabel='$T_{NC}$ (epochs)',
       title='(b) Width vs collapse speed')
ax.grid(alpha=0.3)

# (c) Prediction experiment: fn crossing -> T_NC gap
ax = axes[2]
try:
    df_pred = pd.read_csv('/tmp/prediction_analysis.csv')
    g_gap = df_pred.groupby('width')['gap'].agg(['mean','std'])
    ax.errorbar(g_gap.index, g_gap['mean'], yerr=g_gap['std'],
                fmt='D-', color='#4CAF50', ms=9, capsize=6, lw=2)
    mean_gap = df_pred.gap.mean()
    ax.axhline(mean_gap, color='black', ls='--', lw=1.5,
               label=f'Grand mean = {mean_gap:.0f} ep')
    ax.set_xscale('log', base=2)
    ax.set_xticks(g_gap.index)
    ax.set_xticklabels(g_gap.index)
    ax.set(xlabel='Width', ylabel='Epochs (fn crosses → T_NC)',
           title='(c) Threshold crossing predicts T_NC')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
except FileNotFoundError:
    ax.text(0.5, 0.5, 'No prediction data\n(all DNF?)',
            ha='center', va='center', transform=ax.transAxes)

fig.suptitle('Width Sweep: Scaling Law and Predictive Power of Feature Norm Threshold',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/fig_width_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_width_sweep.png')

# Print paper-ready numbers
print('\n=== PAPER NUMBERS ===')
print(f'Scaling law: fn ∝ width^{slope:.3f}  (R²={r**2:.4f})')
for w in widths.astype(int):
    row = g_fn.loc[w]
    tnc_row = g_tnc.loc[w] if w in g_tnc.index else None
    cv = row['std']/row['mean'] if row['std'] > 0 else 0
    tnc_str = (f"T_NC={tnc_row['mean']:.0f}±{tnc_row['std']:.0f}"
               if tnc_row is not None else 'DNF')
    print(f'  width={w:>4}: fn={row["mean"]:.4f}±{row["std"]:.4f}'
          f'  CV={cv:.3f}  {tnc_str}')


<>:25: SyntaxWarning: invalid escape sequence '\p'
<>:25: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_2225/1316520224.py:25: SyntaxWarning: invalid escape sequence '\p'
  label=f'Fit: fn $\propto$ width$^{{{slope:.2f}}}$\n$R^2={r**2:.3f}$')


Saved: fig_width_sweep.png

=== PAPER NUMBERS ===
Scaling law: fn ∝ width^-0.064  (R²=0.8383)
  width= 128: fn=1.2405±0.1047  CV=0.084  T_NC=383±15
  width= 256: fn=1.1250±0.0406  CV=0.036  T_NC=327±12
  width= 512: fn=1.0958±0.0174  CV=0.016  T_NC=310±10
  width=1024: fn=1.0795±0.0588  CV=0.054  T_NC=257±12


In [9]:
import os
# Download summary CSV and figure
for f in ['/tmp/sweep_width.csv',
           '/tmp/prediction_analysis.csv',
           '/tmp/fig_width_sweep.png']:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded: {f}')

# Per-seed CSVs
for w in [128, 256, 512, 1024]:
    for s in range(3):
        p = f'/tmp/width{w}_s{s}.csv'
        if os.path.exists(p):
            files.download(p)
print('All files downloaded.')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: /tmp/sweep_width.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files downloaded.
